# 1. Tạo thư mục mới để loại bỏ ảnh trùng bên tập valid

In [3]:
import json
import shutil
import hashlib
from pathlib import Path

IMG_EXT = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}

SRC = Path("/Users/mac/Detect_Drill_Bit/original-data")
DST = Path("/Users/mac/Detect_Drill_Bit/clean-data")


def md5(file):
    with open(file, "rb") as f:
        return hashlib.md5(f.read()).hexdigest()


def copy_folder(src, dst):
    if dst.exists():
        shutil.rmtree(dst)
    shutil.copytree(src, dst)


# ============================
# Copy train và test
# ============================

copy_folder(SRC / "train", DST / "train")
copy_folder(SRC / "test", DST / "test")

# ============================
# Lấy toàn bộ hash của Train
# ============================

train_hash = {}

for img in (SRC / "train").rglob("*"):
    if img.suffix.lower() not in IMG_EXT:
        continue

    train_hash[md5(img)] = img

print(f"Train images: {len(train_hash)}")


Train images: 4364


In [4]:
# Đọc annotation
with open(SRC / "valid" / "_annotations.coco.json", "r") as f:
    coco = json.load(f)

valid_root = SRC / "valid"
dst_valid = DST / "valid"

# Tạo index: tên file -> đường dẫn thật
valid_index = {
    p.name: p
    for p in valid_root.rglob("*")
    if p.suffix.lower() in IMG_EXT
}

keep_images = []
keep_ids = set()
removed = []

# Lọc ảnh trùng
for img in coco["images"]:

    filename = Path(img["file_name"]).name
    img_path = valid_index.get(filename)

    if img_path is None:
        continue

    if md5(img_path) in train_hash:
        removed.append(filename)
    else:
        keep_images.append(img)
        keep_ids.add(img["id"])

# Lọc annotation
keep_annotations = [
    ann for ann in coco["annotations"]
    if ann["image_id"] in keep_ids
]

# Tạo thư mục validation mới
if dst_valid.exists():
    shutil.rmtree(dst_valid)
dst_valid.mkdir(parents=True)

# Copy ảnh
for img in keep_images:
    filename = Path(img["file_name"]).name
    src = valid_index[filename]
    dst = dst_valid / src.relative_to(valid_root)

    dst.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(src, dst)

# Lưu annotation mới
coco["images"] = keep_images
coco["annotations"] = keep_annotations

with open(dst_valid / "_annotations.coco.json", "w") as f:
    json.dump(coco, f, indent=2)

print(f"Removed duplicate: {len(removed)}")
print(f"Validation images: {len(keep_images)}")
print(f"Validation annotations: {len(keep_annotations)}")

Removed duplicate: 38
Validation images: 844
Validation annotations: 959


# 2. Kiểm tra data leaky trong data clean

In [5]:
import hashlib
from pathlib import Path

IMG_EXT = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}


def compute_md5(file_path):
    """Tính MD5 của một ảnh"""
    with open(file_path, "rb") as f:
        return hashlib.md5(f.read()).hexdigest()


def get_image_hashes(folder):
    """
    Trả về dict:
        md5 -> đường dẫn ảnh
    """
    hashes = {}

    for img in Path(folder).rglob("*"):
        if img.suffix.lower() not in IMG_EXT:
            continue

        md5 = compute_md5(img)
        hashes[md5] = img

    return hashes


# ==========================
# Đường dẫn dataset
# ==========================

train_dir = "/Users/mac/Detect_Drill_Bit/clean-data/train"
valid_dir = "/Users/mac/Detect_Drill_Bit/clean-data/valid"
test_dir = "/Users/mac/Detect_Drill_Bit/clean-data/test"

# ==========================
# Tính hash
# ==========================

train_hash = get_image_hashes(train_dir)
valid_hash = get_image_hashes(valid_dir)
test_hash = get_image_hashes(test_dir)

# ==========================
# Kiểm tra leakage
# ==========================

train_valid = set(train_hash) & set(valid_hash)
train_test = set(train_hash) & set(test_hash)
valid_test = set(valid_hash) & set(test_hash)

print("=" * 60)
print("DATA LEAKAGE REPORT")
print("=" * 60)

print(f"Train ↔ Valid : {len(train_valid)}")
print(f"Train ↔ Test  : {len(train_test)}")
print(f"Valid ↔ Test  : {len(valid_test)}")

# ==========================
# Hiển thị chi tiết
# ==========================

if train_valid:
    print("\nTrain ↔ Valid")
    for h in train_valid:
        print(f"Train : {train_hash[h]}")
        print(f"Valid : {valid_hash[h]}")
        print("-" * 60)

if train_test:
    print("\nTrain ↔ Test")
    for h in train_test:
        print(f"Train : {train_hash[h]}")
        print(f"Test  : {test_hash[h]}")
        print("-" * 60)

if valid_test:
    print("\nValid ↔ Test")
    for h in valid_test:
        print(f"Valid : {valid_hash[h]}")
        print(f"Test  : {test_hash[h]}")
        print("-" * 60)

if not (train_valid or train_test or valid_test):
    print("\n✅ Không phát hiện data leakage.")

DATA LEAKAGE REPORT
Train ↔ Valid : 0
Train ↔ Test  : 0
Valid ↔ Test  : 0

✅ Không phát hiện data leakage.


# 3. Convert sang định dạng yolo để train nhanh hơn tối ưu cho gpu

In [6]:
from ultralytics.data.converter import convert_coco

ROOT = "/Users/mac/Detect_Drill_Bit/clean-data"

# Chuyển đổi train, valid và test
for split in ["train", "valid"]:
    print(f"Converting {split}...")

    convert_coco(
        labels_dir=f"{ROOT}/{split}",
        save_dir=f"{ROOT}/{split}",
        cls91to80=False
    )

Converting train...
Annotations /Users/mac/Detect_Drill_Bit/clean-data/train/_annotations.coco.json: 100% ━━━━━━━━━━━━ 3597/3597 6.0Kit/s 0.6s0.1s
COCO data converted successfully.
Results saved to /Users/mac/Detect_Drill_Bit/clean-data/train-2
Converting valid...
Annotations /Users/mac/Detect_Drill_Bit/clean-data/valid/_annotations.coco.json: 100% ━━━━━━━━━━━━ 706/706 6.7Kit/s 0.1s0.0s
COCO data converted successfully.
Results saved to /Users/mac/Detect_Drill_Bit/clean-data/valid-2


In [7]:
convert_coco(
    labels_dir=f"{ROOT}/test/Bright_Field",
    save_dir=f"{ROOT}/test/Bright_Field",
    cls91to80=False
)
convert_coco(
    labels_dir=f"{ROOT}/test/Dark_Field",
    save_dir=f"{ROOT}/test/Dark_Field",
    cls91to80=False
)

print("=" * 60)
print("✅ Hoàn thành chuyển đổi COCO → YOLO")
print("=" * 60)

Annotations /Users/mac/Detect_Drill_Bit/clean-data/test/Bright_Field/_annotations.coco.json: 100% ━━━━━━━━━━━━ 545/545 6.3Kit/s 0.1s
COCO data converted successfully.
Results saved to /Users/mac/Detect_Drill_Bit/clean-data/test/Bright_Field-2
Annotations /Users/mac/Detect_Drill_Bit/clean-data/test/Dark_Field/_annotations.coco.json: 100% ━━━━━━━━━━━━ 549/549 6.4Kit/s 0.1s
COCO data converted successfully.
Results saved to /Users/mac/Detect_Drill_Bit/clean-data/test/Dark_Field-2
✅ Hoàn thành chuyển đổi COCO → YOLO


In [8]:
from pathlib import Path
import shutil

SRC = Path("/Users/mac/Detect_Drill_Bit/clean-data/train")
DST = Path("/Users/mac/Detect_Drill_Bit/clean-data/train-2/images")

IMG_EXT = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}

DST.mkdir(parents=True, exist_ok=True)

count = 0

for img in SRC.rglob("*"):
    if img.suffix.lower() in IMG_EXT:
        shutil.copy2(img, DST / img.name)
        count += 1

print(f"Copied {count} images.")

Copied 4364 images.


In [9]:
from pathlib import Path
import shutil

SRC = Path("/Users/mac/Detect_Drill_Bit/clean-data/test/Bright_Field")
DST = Path("/Users/mac/Detect_Drill_Bit/clean-data/test/Bright_Field-2/images")

IMG_EXT = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}

DST.mkdir(parents=True, exist_ok=True)

count = 0

for img in SRC.rglob("*"):
    if img.suffix.lower() in IMG_EXT:
        shutil.copy2(img, DST / img.name)
        count += 1

print(f"Copied {count} images.")

Copied 352 images.


In [10]:
from pathlib import Path
import shutil

SRC = Path("/Users/mac/Detect_Drill_Bit/clean-data/test/Dark_Field")
DST = Path("/Users/mac/Detect_Drill_Bit/clean-data/test/Dark_Field-2/images")

IMG_EXT = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}

DST.mkdir(parents=True, exist_ok=True)

count = 0

for img in SRC.rglob("*"):
    if img.suffix.lower() in IMG_EXT:
        shutil.copy2(img, DST / img.name)
        count += 1

print(f"Copied {count} images.")

Copied 352 images.


In [11]:
from pathlib import Path
import shutil

SRC = Path("/Users/mac/Detect_Drill_Bit/clean-data/valid")
DST = Path("/Users/mac/Detect_Drill_Bit/clean-data/valid-2/images")

IMG_EXT = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}

DST.mkdir(parents=True, exist_ok=True)

count = 0

for img in SRC.rglob("*"):
    if img.suffix.lower() in IMG_EXT:
        shutil.copy2(img, DST / img.name)
        count += 1

print(f"Copied {count} images.")

Copied 844 images.


In [12]:
from pathlib import Path
import shutil

ROOT = Path("/Users/mac/Detect_Drill_Bit/data_convert_yolo_format/test")

for field in ["Bright_Field", "Dark_Field"]:

    outer = ROOT / field / "labels"
    inner = outer / "labels"

    if not inner.exists():
        print(f"{field}: OK")
        continue

    # Di chuyển toàn bộ file txt lên labels/
    for file in inner.glob("*.txt"):
        shutil.move(str(file), outer / file.name)

    # Xóa thư mục labels rỗng
    inner.rmdir()

    print(f"{field}: Done")

print("Hoàn thành.")

Bright_Field: Done
Dark_Field: Done
Hoàn thành.


In [13]:
from pathlib import Path
import shutil

ROOT = Path("/Users/mac/Detect_Drill_Bit/data_convert_yolo_format/train")

outer = ROOT / "labels"
inner = outer / "labels"

if not inner.exists():
    print("Đã đúng cấu trúc.")
else:
    # Move toàn bộ file .txt lên labels/
    for file in inner.glob("*.txt"):
        shutil.move(str(file), outer / file.name)

    # Xóa thư mục labels rỗng
    inner.rmdir()

    print("Đã chuyển labels/labels -> labels")

print("Hoàn thành.")

Đã chuyển labels/labels -> labels
Hoàn thành.


In [14]:
from pathlib import Path
import shutil

ROOT = Path("/Users/mac/Detect_Drill_Bit/data_convert_yolo_format/valid")

outer = ROOT / "labels"
inner = outer / "labels"

if not inner.exists():
    print("Đã đúng cấu trúc.")
else:
    # Move toàn bộ file .txt lên labels/
    for file in inner.glob("*.txt"):
        shutil.move(str(file), outer / file.name)

    # Xóa thư mục labels rỗng
    inner.rmdir()

    print("Đã chuyển labels/labels -> labels")

print("Hoàn thành.")

Đã chuyển labels/labels -> labels
Hoàn thành.
